# Middleware

Middleware provides a way to more tightly control what happens inside the agent . Middleware is useful for the following:
 * Tracking agent behavior with logging, analytics and debugging
 * Transforming prompts, tool selection, and output formatting
 * Adding retries , fallbacks and early termination logic.
 * Applying rate limits, guardrails, and Pll detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
* long running conversations that exceed context windows
* Multi-turn dialogues with extensive history
* Applications where preserving full conversation context matters

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent= create_agent(
    model="groq:openai/gpt-oss-20b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",
            trigger = ("messages",10),
            keep =("messages",4)
        )
        
    ]
)


In [6]:
### Run with a thread is
config={"configurable":{"thread_id":"test-1"}}

In [8]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "what is 100/4?",
    "what is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
    
]
for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content = q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages:{len(response["messages"])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='15966f9d-6a4e-497d-9902-ee6dcede75e1'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'We need to respond to "What is 2+2?" It\'s a simple arithmetic. Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 78, 'total_tokens': 121, 'completion_time': 0.073244953, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.003689231, 'prompt_tokens_details': None, 'queue_time': 0.367693932, 'total_time': 0.076934184}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e11c8bbf69', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1b4a2-fc3a-702a-be57-dc35311b0fca-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 43, 'total_tokens': 121, 'output_token_details': {'reasonin

### TOKEN Size

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

@tool
def search_hotels(city: str)-> str:
    """ Search hotels - returns long response to use more tokens."""
    return f""""Hotels in {city}:
    1.Grand Hotel - 5 start, $350/night , spa, pool, gym
    2.City inn - 4 start, $180/night, business center
    3.Bidget stay - $75/night , free wifi"""

agent =  create_agent(
    model="groq:openai/gpt-oss-20b",
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model="groq:openai/gpt-oss-20b",
        trigger= ("tokens",550),
        keep = ("tokens",200),
    )
                     
 ]
)

config = {"configurable": {"thread_id":"test-1"}}

#token counter(approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content))for m in messages)
    return total_chars // 4 # 4 chars = 1 token
    

In [10]:
#run test
cities = [ "Paris","London","Tokyo","New York","Dubai","Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content = f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response["messages"])} message")
    print(f"{(response['messages'])}")
    


Paris: ~249 tokens, 4 message
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='5411acd6-cd7b-447b-8044-170aeb1a4c77'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user says: "Find hotels in Paris". The system says we have a function search_hotels that takes city: string. So we should call that function.', 'tool_calls': [{'id': 'fc_fa424a9b-64e1-42e2-9b8c-f1603d057ef1', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 129, 'total_tokens': 187, 'completion_time': 0.06656255, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.007304916, 'prompt_tokens_details': None, 'queue_time': 0.047439993, 'total_time': 0.073867466}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_pr

## Fraction


In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str)->str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, city inn $180, Budget stay $75"

# LOW fraction for testing!
aagent =  create_agent(
    model="groq:openai/gpt-oss-20b",
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model="groq:openai/gpt-oss-20b",
        trigger= ("fraction",0.005), # 0.5% = ~640 tokens
        keep = ("tokens",0.002), # 0.2% = ~256 tokens
    )
                     
 ]
)

config = {"configurable": {"thread_id":"test-1"}}

#token counter(approximate)
def count_tokens(messages):
    return sum(len(str(m.content))for m in messages) // 4



In [13]:
#run test
cities = [ "Paris","London","Tokyo","New York","Dubai","Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content = f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    fraction = tokens/128000 
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(f"{(response['messages'])}")

Paris: ~817 tokens (0.6383%), 4 msgs
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT  \nThe user is compiling a comprehensive list of recommended hotels across multiple cities (Paris, London, Tokyo, New\u202fYork, Dubai, and now Singapore). The goal is to later filter these hotels by price, star rating, amenities, check availability for a specified date range, or obtain booking links.\n\n## SUMMARY  \n**Paris**  \n- Grand Hotel – $350/night, 5\u202fstars, spa/pool/gym  \n- City Inn – $180/night, 4\u202fstars, business center  \n- Budget Stay – $75/night, 3\u202fstars, free Wi‑Fi  \n\n**London**  \n- The Savoy – $650/night, 5\u202fstars, historic luxury  \n- Premier Inn London – $120/night, 3\u202fstars, budget‑friendly  \n- The Ritz London – $950/night, 5\u202fstars, iconic luxury  \n- CitizenM London Shoreditch – $200/night, 4\u202fstars, tech‑savvy rooms  \n- The Z Hotel Shoreditch – $90/night, 3\u202fstars, compact rooms  \n\n**Tokyo**  \n-

## Human in the loop MiddleWare

Pause agent execution  for human approval, editing , or rejection of tool calls, before they execute. human in the loop is useful for the following:
* high stakes operations requiring human approval, (e.g database, writes, financial transactions)
* compliance workflows where human oversight is mandatory
* long running conversations where human feedback guides the agent

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

def read_email_tool(email_id: str) ->str:
    """Mock function to read en email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient : str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'" 

In [14]:
agent=create_agent(
    model="groq:openai/gpt-oss-20b",
    tools = [read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware =[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
                
            }
        )
    ]
   
)

In [15]:
config = {"configurable": {"thread_id":"test-approve"}}
#step 1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config
)

In [16]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='76e72c1b-da61-471b-a5ac-75412507fae2'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool with body "How are you?" recipient "john@test.com" subject "Hello".', 'tool_calls': [{'id': 'fc_7600402c-f1c1-4c28-b6ff-d50cd52d067b', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 174, 'total_tokens': 236, 'completion_time': 0.063823002, 'completion_tokens_details': {'reasoning_tokens': 25}, 'prompt_time': 0.00973356, 'prompt_tokens_details': None, 'queue_time': 0.155791887, 'total_time': 0.073556562}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_75c733514d', 'service_tier': 'on_demand', 'finish_reas

In [17]:
# step 2: approve
from langgraph.types import Command
if "_interrupt_" in result:
    print("paused ! approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decision":[
                    {"type":"approve"}
                ]
            }
        ),
        config = config
    )
    
    print(f"Result: {result['message'][-1].content}")

In [19]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='76e72c1b-da61-471b-a5ac-75412507fae2'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool with body "How are you?" recipient "john@test.com" subject "Hello".', 'tool_calls': [{'id': 'fc_7600402c-f1c1-4c28-b6ff-d50cd52d067b', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 174, 'total_tokens': 236, 'completion_time': 0.063823002, 'completion_tokens_details': {'reasoning_tokens': 25}, 'prompt_time': 0.00973356, 'prompt_tokens_details': None, 'queue_time': 0.155791887, 'total_time': 0.073556562}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_75c733514d', 'service_tier': 'on_demand', 'finish_reas

### Reject

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

def read_email_tool(email_id: str) ->str:
    """Mock function to read en email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient : str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'" 

agent=create_agent(
    model="groq:openai/gpt-oss-20b",
    tools = [read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware =[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
                
            }
        )
    ]
   
)

In [21]:
config = {"configurable": {"thread_id":"test-reject"}}
#step 1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config
)

In [22]:
# step 2: reject
from langgraph.types import Command
if "_interrupt_" in result:
    print("paused ! approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decision":[
                    {"type":"reject"}
                ]
            }
        ),
        config = config
    )
    
    print(f"Result: {result['message'][-1].content}")

In [23]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0c0c0ed9-05df-4775-a131-9e1da83c96b5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the send_email_tool.', 'tool_calls': [{'id': 'fc_d139481f-efe3-46b7-ba29-7dee8e455419', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 174, 'total_tokens': 221, 'completion_time': 0.04864587, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.010028656, 'prompt_tokens_details': None, 'queue_time': 0.322833348, 'total_time': 0.058674526}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a4315eb300', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'},

## Editing

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

def read_email_tool(email_id: str) ->str:
    """Mock function to read en email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient : str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'" 

agent=create_agent(
    model="groq:openai/gpt-oss-20b",
    tools = [read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware =[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
                
            }
        )
    ]
   
)

In [24]:
config = {"configurable": {"thread_id":"test-reject"}}
#step 1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content="send email to wrong@email.com with subject 'test' and body 'Hello'")]},
    config = config
)

In [25]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0c0c0ed9-05df-4775-a131-9e1da83c96b5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the send_email_tool.', 'tool_calls': [{'id': 'fc_d139481f-efe3-46b7-ba29-7dee8e455419', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 174, 'total_tokens': 221, 'completion_time': 0.04864587, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.010028656, 'prompt_tokens_details': None, 'queue_time': 0.322833348, 'total_time': 0.058674526}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a4315eb300', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'},

In [28]:
# step 2: Edit and approve
from langgraph.types import Command
if "_interrupt_" in result:
    print("paused ! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decision":[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool",
                            "args": {
                                "recipient":"correct@email.com",
                                "subject":"Corrected Subject",
                                "body":"This was edited by human before sending"
                            }
                        }
                     
                     
                     
                     },
                    
                ]
            }
        ),
        config = config
    )
    
    print(f"Result: {result['message'][-1].content}")

In [29]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0c0c0ed9-05df-4775-a131-9e1da83c96b5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the send_email_tool.', 'tool_calls': [{'id': 'fc_d139481f-efe3-46b7-ba29-7dee8e455419', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 174, 'total_tokens': 221, 'completion_time': 0.04864587, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.010028656, 'prompt_tokens_details': None, 'queue_time': 0.322833348, 'total_time': 0.058674526}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a4315eb300', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'},